In [2]:
import torch
from src.cocoruta_sayless import print_env_config, get_tokenizer_and_model, query_model, say_less, build_prompt

%load_ext autoreload
%autoreload 2

/home/marcos.moretti/repos/conformal-factual-lm/venv-cf-2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print_env_config()

cuda available: True
cuda devices: 4
NVIDIA RTX 4000 Ada Generation
NVIDIA RTX 4000 Ada Generation
NVIDIA RTX 4000 Ada Generation
NVIDIA RTX 4000 Ada Generation
torch cuda version: 12.8
2.10.0+cu128


In [4]:
model_id = "meta-llama/Llama-3.1-8B" #"felipeoes/cocoruta-7b"
# using the second GPU only
tokenizer, model = get_tokenizer_and_model(
    model_id=model_id, device_map="cuda:1", torch_dtype=torch.float16)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 211.13it/s, Materializing param=model.norm.weight]                              


In [49]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que é a Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            max_length=256,
                            temperature=1e-8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)

 A Amazônia Azul é um projeto que visa a preservação da biodiversidade marinha da Amazônia, que é uma das regiões mais ricas em biodiversidade do mundo. O projeto envolve a criação de uma área protegida no mar, que abrange uma área de 1,5 milhão de quilômetros quadrados, e a criação de uma rede de parques marinhos e reservas naturais. O objetivo do projeto é proteger a biodiversidade marinha da Amazônia, que é ameaçada por atividades como a pesca predatória, a poluição e o desmatamento. O projeto também visa a conservação dos recursos naturais da região, como a pesca e a produção de alimentos, e a promoção do desenvolvimento sustentável da região.<|end_of_text|>


In [50]:
# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text)

 A Amazônia Azul é um projeto que visa a preservação da biodiversidade marinha da Amazônia, que é uma das regiões mais ricas em biodiversidade do mundo. O projeto envolve a criação de uma área protegida no mar, que abrange uma área de 1,5 milhão de quilômetros quadrados, e a criação de uma rede de parques marinhos e reservas naturais. O objetivo do projeto é proteger a biodiversidade marinha da Amazônia, que é ameaçada por atividades como a pesca predatória, a poluição e o desmatamento. O projeto também visa a conservação dos recursos naturais da região, como a pesca e a produção de alimentos, e a promoção do desenvolvimento sustentável da região.


In [51]:
tokenizer.decode(_[0], skip_special_tokens=True)

'### Pergunta: O que é a Amazônia Azul?\n### Resposta: A Amazônia Azul é um projeto que visa a preservação da biodiversidade marinha da Amazônia, que é uma das regiões mais ricas em biodiversidade do mundo. O projeto envolve a criação de uma área protegida no mar, que abrange uma área de 1,5 milhão de quilômetros quadrados, e a criação de uma rede de parques marinhos e reservas naturais. O objetivo do projeto é proteger a biodiversidade marinha da Amazônia, que é ameaçada por atividades como a pesca predatória, a poluição e o desmatamento. O projeto também visa a conservação dos recursos naturais da região, como a pesca e a produção de alimentos, e a promoção do desenvolvimento sustentável da região.'

In [52]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            max_length=256,
                            temperature=1e-8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)

 A legislação brasileira não possui uma lei específica para a preservação dos recifes de corais na Amazônia Azul. No entanto, a Lei nº 9.985, de 18 de julho de 2000, que instituiu o Sistema Nacional de Unidades de Conservação da Natureza (SNUC), estabelece que as unidades de conservação marinha devem ser criadas com o objetivo de proteger os ecossistemas marinhos e costeiros, incluindo os recifes de corais. Além disso, a Lei nº 11.516, de 28 de agosto de 2007, que instituiu a Política Nacional do Meio Ambiente, estabelece que as ações de conservação e preservação do meio ambiente devem ser realizadas de forma integrada, considerando os diferentes ecossistemas e as interações entre eles. Portanto, a preservação dos recifes de cor


In [53]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            #streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            max_length=256,
                            temperature=1e-8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)

In [54]:
# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text)

 A legislação brasileira não possui uma lei específica para a preservação dos recifes de corais na Amazônia Azul. No entanto, a Lei nº 9.985, de 18 de julho de 2000, que instituiu o Sistema Nacional de Unidades de Conservação da Natureza (SNUC), estabelece que as unidades de conservação marinha devem ser criadas com o objetivo de proteger os ecossistemas marinhos e costeiros, incluindo os recifes de corais. Além disso, a Lei nº 11.516, de 28 de agosto de 2007, que instituiu a Política Nacional do Meio Ambiente, estabelece que as ações de conservação e preservação do meio ambiente devem ser realizadas de forma integrada, considerando os diferentes ecossistemas e as interações entre eles. Portanto, a preservação dos recifes de cor


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            #streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            #max_length=256,
                            max_new_tokens=1000,
                            temperature=1e-8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)

In [56]:
# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text)

 A legislação brasileira não possui uma lei específica para a preservação dos recifes de corais na Amazônia Azul. No entanto, a Lei nº 9.985, de 18 de julho de 2000, que instituiu o Sistema Nacional de Unidades de Conservação da Natureza (SNUC), estabelece que as unidades de conservação marinha devem ser criadas com o objetivo de proteger os ecossistemas marinhos e costeiros, incluindo os recifes de corais. Além disso, a Lei nº 11.516, de 28 de agosto de 2007, que instituiu a Política Nacional do Meio Ambiente, estabelece que as ações de conservação e preservação do meio ambiente devem ser realizadas de forma integrada, considerando os diferentes ecossistemas e as interações entre eles. Portanto, a preservação dos recifes de corais na Amazônia Azul deve ser tratada como parte da conservação e preservação do meio ambiente em geral, e deve ser abordada de forma integrada com as demais ações de conservação e preservação do meio ambiente na região.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(
            input_ids,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=1000,
            num_return_sequences=1,
            stopping_criteria=[StopOnString(stop_string, input_text)],
            do_sample=False,
)

# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text)

 A legislação brasileira não possui uma lei específica para a preservação dos recifes de corais na Amazônia Azul. No entanto, a Lei nº 9.985, de 18 de julho de 2000, que instituiu o Sistema Nacional de Unidades de Conservação da Natureza (SNUC), estabelece que as unidades de conservação marinha devem ser criadas com o objetivo de proteger os ecossistemas marinhos e costeiros, incluindo os recifes de corais. Além disso, a Lei nº 11.516, de 28 de agosto de 2007, que instituiu a Política Nacional do Meio Ambiente, estabelece que as ações de conservação e preservação do meio ambiente devem ser realizadas de forma integrada, considerando os diferentes ecossistemas e as interações entre eles. Portanto, a preservação dos recifes de corais na Amazônia Azul deve ser tratada como parte da conservação e preservação do meio ambiente em geral, e deve ser abordada de forma integrada com as demais ações de conservação e preservação do meio ambiente na região.


In [59]:
output_text1 = output_text

In [64]:
output_text2 = output_text

In [65]:
output_text1 == output_text2

True

In [ ]:
output_text1

' A legislação brasileira não possui uma lei específica para a preservação dos recifes de corais na Amazônia Azul. No entanto, a Lei nº 9.985, de 18 de julho de 2000, que instituiu o Sistema Nacional de Unidades de Conservação da Natureza (SNUC), estabelece que as unidades de conservação marinha devem ser criadas com o objetivo de proteger os ecossistemas marinhos e costeiros, incluindo os recifes de corais. Além disso, a Lei nº 11.516, de 28 de agosto de 2007, que instituiu a Política Nacional do Meio Ambiente, estabelece que as ações de conservação e preservação do meio ambiente devem ser realizadas de forma integrada, considerando os diferentes ecossistemas e as interações entre eles. Portanto, a preservação dos recifes de corais na Amazônia Azul deve ser tratada como parte da conservação e preservação do meio ambiente em geral, e deve ser abordada de forma integrada com as demais ações de conservação e preservação do meio ambiente na região.'

In [76]:
question1 = "O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?"
prompt1 = build_prompt(question1)
print(prompt1)
output_text1 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0, n_samples=1)
output_text2 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0, n_samples=1)
output_text3 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0, n_samples=1)
print(output_text1)
print(output_text2)
print(output_text3)
print(output_text1 == output_text2)
print(output_text1 == output_text3)

### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?
### Resposta:
 A legislação brasileira não possui uma lei específica para a preservação dos recifes de corais na Amazônia Azul. No entanto, a Lei nº 9.985, de 18 de julho de 2000, que instituiu o Sistema Nacional de Unidades de Conservação da Natureza (SNUC), estabelece que as unidades de conservação marinha devem ser criadas com o objetivo de proteger os ecossistemas marinhos e costeiros, incluindo os recifes de corais. Além disso, a Lei nº 11.516, de 28 de agosto de 2007, que instituiu a Política Nacional do Meio Ambiente, estabelece que as ações de conservação e preservação do meio ambiente devem ser realizadas de forma integrada, considerando os diferentes ecossistemas e as interações entre eles. Portanto, a preservação dos recifes de corais na Amazônia Azul deve ser tratada como parte da conservação e preservação do meio ambiente em geral, e deve ser abordad

In [77]:
question1 = "O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?"
prompt1 = build_prompt(question1)
print(prompt1)
output_text1 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0.8, n_samples=1)
output_text2 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0.8, n_samples=1)
output_text3 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0.8, n_samples=1)
print(output_text1)
print(output_text2)
print(output_text3)
print(output_text1 == output_text2)
print(output_text1 == output_text3)

### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?
### Resposta:
 A legislação não tem nenhuma norma específica sobre a preservação dos recifes de corais. A única legislação que trata especificamente sobre o meio marinho é a Lei 9.605/98, que tem como objetivo a proteção do meio ambiente. Nesta lei, encontramos um dispositivo específico sobre o uso do corais marinhos, no Art. 42, § 1º: "§ 1º - A coleta e a exploração de corais marinhos e outros organismos bentônicos, para fins científicos ou didáticos, dependerão de autorização da autoridade competente." Sendo assim, o uso e a exploração dos recifes de corais somente poderá ser feito com autorização prévia da autoridade competente.

###
 
A Lei nº 12.715, de 2012, conhecida como Lei de Proteção dos Recifes de Corais, foi aprovada pelo Congresso Nacional em 2012, após 19 anos de discussão. Ela tem como objetivo proteger os recifes de corais, considerados um dos ec

In [6]:
question2 = "O que é a Amazônia Azul?"
prompt2 = build_prompt(question2)
print(prompt2)
output_text1 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0, n_samples=1)
output_text2 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0, n_samples=1)
output_text3 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0, n_samples=1)
print(output_text1)
print(output_text2)
print(output_text3)
print(output_text1 == output_text2)
print(output_text1 == output_text3)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


### Pergunta: O que é a Amazônia Azul?
### Resposta:
 A Amazônia Azul é um projeto que visa a preservação da biodiversidade marinha da Amazônia, que é uma das regiões mais ricas em biodiversidade do mundo. O projeto envolve a criação de uma área protegida no mar, que abrange uma área de 1,5 milhão de quilômetros quadrados, e a criação de uma rede de parques marinhos e reservas naturais. O objetivo do projeto é proteger a biodiversidade marinha da Amazônia, que é ameaçada por atividades como a pesca predatória, a poluição e o desmatamento. O projeto também visa a conservação dos recursos naturais da região, como a pesca e a produção de alimentos, e a promoção do desenvolvimento sustentável da região.
 A Amazônia Azul é um projeto que visa a preservação da biodiversidade marinha da Amazônia, que é uma das regiões mais ricas em biodiversidade do mundo. O projeto envolve a criação de uma área protegida no mar, que abrange uma área de 1,5 milhão de quilômetros quadrados, e a criação de uma 

In [7]:
question2 = "O que é a Amazônia Azul?"
prompt2 = build_prompt(question2)
print(prompt2)
output_text1 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0.8, n_samples=1)
output_text2 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0.8, n_samples=1)
output_text3 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0.8, n_samples=1)
print(output_text1)
print(output_text2)
print(output_text3)
print(output_text1 == output_text2)
print(output_text1 == output_text3)

### Pergunta: O que é a Amazônia Azul?
### Resposta:
 A Amazônia Azul é a parte da Amazônia que fica na costa atlântica, na região das ilhas.

###
 A Amazônia Azul é o conjunto de rios que estão dentro da Amazônia e que formam o maior sistema de rios do mundo.
 A Amazônia Azul é uma área de 3 milhões de km2, que corresponde a 12% do oceano Atlântico Sul. Nesta área, as águas são mais quentes, mais salgadas e mais ricas em nutrientes do que as demais regiões oceânicas. Essas águas, que estão entre os mais ricos ecossistemas do planeta, são responsáveis por 20% da produção mundial de peixes.
###
False
False


In [6]:
prompt = "### Pergunta: O que é a Amazônia Azul?\n### Resposta:"
output = query_model(model, tokenizer, prompt)
output

'### Pergunta: O que é a Amazônia Azul?\n### Resposta: A Amazônia Azul é um projeto que visa a preservação da biodiversidade marinha da Amazônia, que é uma das regiões mais ricas em biodiversidade do mundo. O projeto envolve a criação de uma área protegida no mar, que abrange uma área de 1,5 milhão de quilômetros quadrados, e a criação de uma rede de parques marinhos e reservas naturais. O objetivo do projeto é proteger a biodiversidade marinha da Amazônia, que é ameaçada por atividades como a pesca predatória, a poluição e o desmatamento. O projeto também visa a conservação dos recursos naturais da região, como a pesca e a produção de alimentos, e a promoção do desenvolvimento sustentável da região.'

In [7]:
prompt = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
output = query_model(model, tokenizer, prompt)
output

'### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta: A legislação brasileira não possui uma lei específica para a preservação dos recifes de corais na Amazônia Azul. No entanto, a Lei nº 9.985, de 18 de julho de 2000, que instituiu o Sistema Nacional de Unidades de Conservação da Natureza (SNUC), estabelece que as unidades de conservação marinha devem ser criadas com o objetivo de proteger os ecossistemas marinhos e costeiros, incluindo os recifes de corais. Além disso, a Lei nº 11.516, de 28 de agosto de 2007, que instituiu a Política Nacional do Meio Ambiente, estabelece que as ações de conservação e preservação do meio ambiente devem ser realizadas de forma integrada, considerando os diferentes ecossistemas e as interações entre eles. Portanto, a preservação dos recifes de corais na Amazônia Azul deve ser tratada como parte da conservação e preservação do meio ambiente em geral, e deve ser aborda

In [ ]:
# Copied threshold from /factscore_a=1_alpha=0.15_conf=frequency+gpt.txt.
# Compute new ones by running factscore.py with desired parameters and setting compute_single_threshold=True
threshold = 4.8998812119930735
merged_output, (accepted_subclaims, all_subclaims) = say_less(
    model, tokenizer, prompt, output, threshold
)
print("Original output: ")
print(output)
print("\n\n\n\n\nModified output: ")
print(merged_output)
print("\n\n\n\nAccepted sub-claims: ")
print(accepted_subclaims)
print("\n\n\n\nAll sub-claims: ")
print(all_subclaims)

Expecting value: line 1 column 5 (char 4)
Failed to parse as jsonl

    Você é um assistente prestativo cuja função é dividir suas entradas em um conjunto de pequenas afirmações, para que um ser humano possa verificar facilmente cada uma delas. Certifique-se de que cada afirmação seja pequena e não sobreposta às demais.

    Aqui está a entrada que você precisa dividir:
    Por favor, divida a seguinte entrada em um conjunto de pequenas afirmações independentes e retorne a saída no formato jsonl, onde cada linha seja {subclaim:[AFIRMAÇÃO], cocoruta-score:[CONF]}. A pontuação de confiança [CONF] deve representar o seu nível de confiança na afirmação, onde 1 corresponde a fatos e resultados óbvios, como 'A Terra é redonda' e '1+1=2'. Já 0 corresponde a afirmações muito obscuras ou difíceis de qualquer pessoa saber, como a data de aniversário de pessoas não públicas. A entrada é: 
Você é um assistente prestativo cuja função é dividir suas entradas em um conjunto de pequenas afirmações, pa

TypeError: object of type 'NoneType' has no len()

In [16]:
prompt = f"""
Você é um assistente prestativo cuja função é dividir suas entradas em um conjunto de pequenas afirmações, para que um ser humano possa verificar facilmente cada uma delas. Certifique-se de que cada afirmação seja pequena e não sobreposta às demais.

Aqui está a entrada que você precisa dividir:
Por favor, divida a seguinte entrada em um conjunto de pequenas afirmações independentes e retorne a saída no formato jsonl, onde cada linha seja {{subclaim:[AFIRMAÇÃO], cocoruta-score:[CONF]}}. A pontuação de confiança [CONF] deve representar o seu nível de confiança na afirmação, onde 1 corresponde a fatos e resultados óbvios, como 'A Terra é redonda' e '1+1=2'. Já 0 corresponde a afirmações muito obscuras ou difíceis de qualquer pessoa saber, como a data de aniversário de pessoas não públicas. A entrada é: {prompt}

Por favor, forneça sua resposta:
"""
print(prompt)
output = query_model(model, tokenizer, prompt, max_tokens=1000, temperature=0)
print(output)


Você é um assistente prestativo cuja função é dividir suas entradas em um conjunto de pequenas afirmações, para que um ser humano possa verificar facilmente cada uma delas. Certifique-se de que cada afirmação seja pequena e não sobreposta às demais.

Aqui está a entrada que você precisa dividir:
Por favor, divida a seguinte entrada em um conjunto de pequenas afirmações independentes e retorne a saída no formato jsonl, onde cada linha seja {subclaim:[AFIRMAÇÃO], cocoruta-score:[CONF]}. A pontuação de confiança [CONF] deve representar o seu nível de confiança na afirmação, onde 1 corresponde a fatos e resultados óbvios, como 'A Terra é redonda' e '1+1=2'. Já 0 corresponde a afirmações muito obscuras ou difíceis de qualquer pessoa saber, como a data de aniversário de pessoas não públicas. A entrada é: 
Você é um assistente prestativo cuja função é dividir suas entradas em um conjunto de pequenas afirmações, para que um ser humano possa verificar facilmente cada uma delas. Certifique-se d